In [1]:
# the parameters from MCMC results to form the parameters file for simulations

import pandas as pd

def read_nml_file(file_path):
    """ read nml file content """
    with open(file_path, "r") as file:
        lines = file.readlines()
    return lines

def parse_nml_lines(lines):
    """ parse the lines in a nml-format with multiple groups """
    data = {}
    current_group = None
    for line in lines:
        line = line.strip()
        if line.startswith('&'): # start a new group
            current_group = line[1:]
            data[current_group] = {}
        elif line.startswith('/'): # end current group
            current_group = None
        elif "=" in line and current_group:
            key, value = line.split('=', 1)
            data[current_group][key.strip()] = value.strip()
    return data

def update_nml_data(data, group, updates):
    """ update the nml date for the specified group, where the updates is a dictionary containing the key-value pairs to be updated  """
    if group in data:
        for key, value in updates.items():
            if key in data[group]:
                data[group][key] = value

def write_nml_file(file_path, data):
    """ re-write the updated nml data back to the file, keeping the group structure """
    with open(file_path, 'w') as file:
        for group, items in data.items():
            file.write(f'&{group}\n')
            for key, value in items.items():
                file.write(f'{key} = {value}\n')
            file.write('/\n')


def run_params4nml(file_params, org_data, file_path):
    # read the parameter file 
    df_dat = pd.read_csv(file_params).iloc[-1]
    print("test len: ", len(df_dat))
    print(df_dat)

    df_dict = {}
    for idx in df_dat.index:
        if idx.split("_")[-1].rstrip() in ["Tree", "Shrub", "Sphagnum"]:
            item,_,_ = idx.rpartition('_')
            if item == "SLA": item = "SLAx"
            if item in df_dict.keys():
                df_dict[item] = df_dict[item] + ", " + str(df_dat[idx])
            else:
                df_dict[item] = str(df_dat[idx])
        else:
            df_dict[idx] = str(df_dat[idx])
    print(df_dict)
    # update
    i = 0
    nml_name = ["nml_site_params", "nml_species_params"]
    # group_to_update = 'nml_species_params'  # 
    for group_to_update in nml_name:
        for ikey, item in df_dict.items():
            updates = {ikey.strip(): item}  # the content for updating
            update_nml_data(org_data, group_to_update, updates)
            # print(updates)
    # print(i)

    write_nml_file(file_path, org_data)



ls_plots = ["P04", "P06", "P08", "P10", "P11", "P13", "P16", "P17", "P19", "P20"]

org_file = "parameters.nml" # destination file for nml-format
# ls_plots = ["P04", "P06", "P08", "P10", "P11", "P13", "P16",  "P19", "P20"]
ls_plots = ["P11", "P16", "P17"]

# each plot
for idx_plot, iplot in enumerate(ls_plots):
    print(iplot)
    file_params = f"../../outputs/run_mcmc_treat_{iplot}/results_mcmc/total_parameter_sets_005.csv"
    file_path   = f"../inputs/in_treat/{iplot}/parameters.nml"#f"inputs/in_pretreat/{iplot}/parameters.nml"
    lines = read_nml_file(file_path)
    data  = parse_nml_lines(lines)
    run_params4nml(file_params, data, file_path)

P11


FileNotFoundError: [Errno 2] No such file or directory: '../inputs/in_treat/P11/parameters.nml'

In [2]:
import os
import f90nml

# update the parameter values in ["nml_site_params", "nml_species_params"] to mcmc_conf

def handle_data(iplot):
    # Define file paths
    parameters_nml_path = f"../inputs/in_treat/{iplot}/parameters.nml" #f"inputs/in_treat/{iplot}/parameters.nml"
    mcmc_conf_nml_path  = f"../configs/conf_treat/mcmc_conf_{iplot}.nml" #"mcmc_conf_P06.nml" #"configs/mcmc_conf.nml"

    # Read parameters.nml content
    nml_params_data    = f90nml.read(parameters_nml_path)
    nml_mcmc_conf_data = f90nml.read(mcmc_conf_nml_path)

    # dict paramter
    dict_site_params = dict(nml_params_data["nml_site_params"])
    dict_spp_params  = dict(nml_params_data["nml_species_params"])

    # modify mcmc_conf data
    for idx_nml, inml in enumerate(nml_mcmc_conf_data["siteDAParams"]["st"]):
        if inml["parname"].lower() in dict_site_params.keys():
            nml_mcmc_conf_data["siteDAParams"]["st"][idx_nml]["parVal"] = dict_site_params[inml["parname"].lower()]

    for idx_sp, isp in enumerate(nml_mcmc_conf_data["spDAParams"]["sp"]):
        for idx_var, ivar in enumerate(isp["var"]):
            if ivar["parname"].lower() in dict_spp_params.keys():
                # if idx_sp < len(nml_mcmc_conf_data["spDAParams"]["sp"]):
                #     print(f"len(nml_mcmc_conf_data['spDAParams']['sp'][idx_sp]['var']): {len(nml_mcmc_conf_data['spDAParams']['sp'][idx_sp]['var'])}")
                print(ivar["parname"].lower(),dict_spp_params[ivar["parname"].lower()])
                nml_mcmc_conf_data["spDAParams"]["sp"][idx_sp]["var"][idx_var]["parval"] = dict_spp_params[ivar["parname"].lower()][idx_sp]

    # write updated mcmc_conf to file
    nml_mcmc_conf_data.write(mcmc_conf_nml_path, force=True)#, mode="w")

ls_plots = ["P04", "P06", "P08", "P10", "P11", "P13", "P16", "P17", "P19", "P20"]
# ls_plots = ["P08"]
for idx, iplot in enumerate(ls_plots):
    print(iplot)
    handle_data(iplot)

P04
laimax [7.442, 7.125, 7.323]
laimin [0.3, 0.3, 0.3]
stom_n [2, 2, 2]
saps [0.3915, 0.4717, 0.4715]
sapr [1.0, 1.0, 1.0]
slax [14.09, 29.48, 128.8]
glmax [41.25, 42.69, 41.86]
grmax [26.06, 18.82, 72.15]
gsmax [9.678, 24.16, 34.0]
alpha [0.385, 0.385, 0.385]
vcmax0 [35.41, 63.69, 75.31]
ds0 [2000, 2000, 2000]
xfang [0, 0, 0]
rdepth [150, 150, 150]
rootmax [500, 500, 500]
stemmax [1000, 1000, 1000]
tau_leaf [1.163, 0.5273, 0.341]
tau_stem [55.64, 3.279, 0.3243]
tau_root [0.5973, 1.672, 0.1562]
q10 [3.072, 2.175, 1.486]
rl0 [36.08, 33.19, 73.03]
rs0 [7.273, 9.391, 63.25]
rr0 [25.23, 12.81, 32.35]
jv [2.828, 0.825, 0.41]
entrpy [667.2, 628.7, 652.4]
gddonset [99.11, 123.9, 155.4]
hmax [24.19, 24.19, 24.19]
hl0 [0.00019, 0.00019, 0.00019]
laimax0 [8.0, 8.0, 8.0]
la0 [0.2, 0.2, 0.2]
fn2l [0.2046, 0.5244, 0.3178]
fn2r [0.6754, 0.1332, 0.291]
s_cleaf [5.107, 4.377, 5.061]
s_cstem [0.9637, 2.603, 6.144]
s_croot [4.038, 0.7037, 1.299]
s_nsc [3.239, 1.834, 0.4476]
s_nsn [1.036, 3.959, 3.485]
